<a href="https://colab.research.google.com/github/jmoussa99/football-ai/blob/main/track_players.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Player Tracking: YOLOv8 + DeepSORT + Team Clustering + Jersey OCR

Detects players in a sports video, tracks them across frames with consistent IDs,
splits them into two teams by jersey color, and reads jersey numbers with OCR.

Run the cells top to bottom. Set your video path and options in **Config** below,
then run **Main loop**. The annotated video saves to `OUTPUT_PATH`, and you can
preview it inline in the last cell.


## 1. Install dependencies
Run once per environment.

In [ ]:
!pip install ultralytics deep-sort-realtime opencv-python easyocr scikit-learn numpy --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 6.2 MB/s eta 0:00:00


## 2. Imports

In [ ]:
from collections import defaultdict, Counter

import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from sklearn.cluster import KMeans


## 3. Config
Edit these instead of passing command-line flags.


In [ ]:
INPUT_PATH = "game.mp4"            # path to your input video
OUTPUT_PATH = "tracked.mp4"        # where the annotated video will be saved

YOLO_MODEL = "yolov8n.pt"          # n/s/m/l/x -- larger = more accurate, slower
CONF_THRESHOLD = 0.4               # detection confidence threshold
MAX_AGE = 30                       # frames to keep a lost track alive

TEAM_CALIB_FRAMES = 60             # frames used to fit the 2-team color model
                                    # (both teams should be visible during this window)

ENABLE_OCR = True                  # jersey-number reading -- set False to skip (much faster)
OCR_EVERY_N_FRAMES = 5             # run OCR this often per track (OCR is the slow part)
USE_GPU = True                     # set False if you don't have a CUDA GPU

SHOW_LIVE_PREVIEW = False          # OpenCV preview window -- usually leave False in a notebook


## 4. Helper classes
Team-color clustering and jersey-number OCR with running vote.

In [ ]:
def dominant_jersey_color(frame, box, torso_frac=(0.15, 0.55)):
    """
    Sample the torso region of a player's bounding box (avoids head/shorts/shoes/background)
    and return its dominant color in HSV as a small feature vector.
    """
    l, t, r, b = box
    h = b - t
    y1 = t + int(h * torso_frac[0])
    y2 = t + int(h * torso_frac[1])
    crop = frame[max(y1, 0):max(y2, 1), max(l, 0):max(r, 1)]
    if crop.size == 0:
        return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    med = np.median(hsv.reshape(-1, 3), axis=0)  # median is robust against edge/background bleed
    return med


class TeamClassifier:
    """
    Collects jersey-color samples for the first N frames, fits a 2-cluster k-means model,
    then assigns any subsequent player crop to Team A or Team B by nearest cluster center.
    """
    def __init__(self, calib_frames=60):
        self.calib_frames = calib_frames
        self.samples = []
        self.model = None
        self.ready = False

    def add_sample(self, color_vec):
        if not self.ready and color_vec is not None:
            self.samples.append(color_vec)

    def maybe_fit(self, frame_idx):
        if not self.ready and frame_idx >= self.calib_frames and len(self.samples) >= 10:
            X = np.array(self.samples)
            self.model = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
            self.ready = True

    def classify(self, color_vec):
        if not self.ready or color_vec is None:
            return None
        label = self.model.predict([color_vec])[0]
        return "Team A" if label == 0 else "Team B"


class JerseyOCR:
    """
    Wraps EasyOCR and keeps a running vote per track ID, since OCR on a single blurry
    frame is unreliable but a majority vote over many frames converges on the real number.
    """
    def __init__(self, enabled=True, gpu=True):
        self.enabled = enabled
        self.reader = None
        if enabled:
            import easyocr
            self.reader = easyocr.Reader(["en"], gpu=gpu)
        self.votes = defaultdict(Counter)

    def read_number(self, frame, box):
        if not self.enabled:
            return None
        l, t, r, b = box
        h = b - t
        y1, y2 = t + int(h * 0.10), t + int(h * 0.55)  # upper-torso band, where numbers usually sit
        crop = frame[max(y1, 0):max(y2, 1), max(l, 0):max(r, 1)]
        if crop.size == 0:
            return None
        crop = cv2.resize(crop, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
        results = self.reader.readtext(crop, allowlist="0123456789")
        digits = "".join(text for _, text, conf in results if conf > 0.4)
        return digits if digits.isdigit() and 0 < len(digits) <= 2 else None

    def update(self, track_id, frame, box, frame_idx, every_n):
        if not self.enabled or frame_idx % every_n != 0:
            return
        num = self.read_number(frame, box)
        if num:
            self.votes[track_id][num] += 1

    def best_guess(self, track_id):
        if track_id not in self.votes or not self.votes[track_id]:
            return "?"
        return self.votes[track_id].most_common(1)[0][0]


## 5. Set up the detector, tracker, team classifier, and OCR

In [ ]:
detector = YOLO(YOLO_MODEL)

tracker = DeepSort(
    max_age=MAX_AGE,
    n_init=3,
    nms_max_overlap=1.0,
    max_cosine_distance=0.3,
    embedder="mobilenet",
    half=True,
    bgr=True,
)

team_clf = TeamClassifier(calib_frames=TEAM_CALIB_FRAMES)
ocr = JerseyOCR(enabled=ENABLE_OCR, gpu=USE_GPU)

print("Detector, tracker, team classifier, and OCR are ready.")


## 6. Main loop
Processes the video frame by frame: detect players, update tracks, classify team by
jersey color, vote on jersey number via OCR, and draw everything onto the output video.

This can take a while depending on video length and whether OCR is enabled -- progress
prints every 100 frames.


In [ ]:
cap = cv2.VideoCapture(INPUT_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (w, h))

team_colors = {"Team A": (255, 100, 0), "Team B": (0, 200, 255), None: (150, 150, 150)}
track_history = defaultdict(list)
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1

    # --- Detect ---
    results = detector(frame, verbose=False)[0]
    detections = []
    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        if cls_id != 0 or conf < CONF_THRESHOLD:   # COCO class 0 = "person"
            continue
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        detections.append(([x1, y1, x2 - x1, y2 - y1], conf, "player"))

    # --- Track ---
    tracks = tracker.update_tracks(detections, frame=frame)

    # --- Calibrate team-color model during warm-up window ---
    if not team_clf.ready:
        for track in tracks:
            if not track.is_confirmed():
                continue
            l, t, r, b = map(int, track.to_ltrb())
            color_vec = dominant_jersey_color(frame, (l, t, r, b))
            team_clf.add_sample(color_vec)
        team_clf.maybe_fit(frame_idx)

    # --- Draw + annotate each confirmed track ---
    for track in tracks:
        if not track.is_confirmed():
            continue
        track_id = track.track_id
        l, t, r, b = map(int, track.to_ltrb())

        color_vec = dominant_jersey_color(frame, (l, t, r, b))
        team = team_clf.classify(color_vec)
        box_color = team_colors[team]

        ocr.update(track_id, frame, (l, t, r, b), frame_idx, OCR_EVERY_N_FRAMES)
        jersey_num = ocr.best_guess(track_id)

        cx, cy = (l + r) // 2, (t + b) // 2
        track_history[track_id].append((cx, cy))

        label = f"#{jersey_num} ID{track_id}" if ENABLE_OCR else f"ID {track_id}"
        if team:
            label += f" ({team})"

        cv2.rectangle(frame, (l, t), (r, b), box_color, 2)
        cv2.putText(frame, label, (l, max(t - 8, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, box_color, 2)

        pts = track_history[track_id][-30:]
        for i in range(1, len(pts)):
            cv2.line(frame, pts[i - 1], pts[i], box_color, 2)

    out.write(frame)

    if SHOW_LIVE_PREVIEW:
        cv2.imshow("Player Tracking", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    if frame_idx % 100 == 0:
        status = "calibrating teams..." if not team_clf.ready else "tracking"
        print(f"Processed {frame_idx} frames ({status})")

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"\nDone. Saved to {OUTPUT_PATH}")


## 7. Jersey-number results
Final majority-vote guess per track ID.

In [ ]:
if ENABLE_OCR:
    print("Final jersey-number guesses per track ID:")
    for tid in sorted(ocr.votes.keys(), key=lambda x: int(x)):
        print(f"  ID {tid}: #{ocr.best_guess(tid)}  (votes: {dict(ocr.votes[tid])})")
else:
    print("OCR was disabled (ENABLE_OCR = False).")


## 8. Preview the annotated video inline
Renders the output video directly in the notebook (works in Jupyter/JupyterLab; on
Colab use the alternate cell below instead).


In [ ]:
from IPython.display import Video

Video(OUTPUT_PATH, embed=True, width=640)


### Alternate: preview in Google Colab
Colab can't always play local mp4 files with the cell above. If you're in Colab, run
this instead.


In [ ]:
# Uncomment if running in Google Colab:

# from IPython.display import HTML
# from base64 import b64encode
#
# with open(OUTPUT_PATH, "rb") as f:
#     video_b64 = b64encode(f.read()).decode()
# HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{video_b64}" type="video/mp4"></video>')
